# Neural Networks - Multi-Layer Network

> Why does modern NNs have thousands of layers?

## 1. Function Composition: The Essence of Depth

- Composition of functions.

## 2. Hidden Representations

- Transform representation of data.
- Hidden neurons = feature detectors.
- Latent representations -> Latent space.

## 3. Why Deep Beats Shallow

- Hierarchical, like human brain.
- Depth -> Reuse (dynamic programming).
- Solve XOR.

## 4. Forward Propagation Through Multiple Layers

- Every layer outputs an activation.
- At layer 0: $a^{(0)} = x$
- At k-th layer:

```math
z^{(1)} = W^{(1)}a^{(0)} + b^{(1)} \\
a^{(1)} = f(z^{(1)})
```

- Output layer:

```math
z^{(L)} = W^{(1)}a^{(L-1)} + b^{(L)} \\
\^{y} = g(z^{(L)})
```

- Weight's shape = (current neurons, previous neurons)
- Batch version:

```math
Z^{(l)} = A^{(l-1)}(W^{(l)})^T + b^{(l)}
```

In [7]:
import numpy as np

class Linear:

    def __init__(self, in_features, out_features):
        self.W = np.random.randn(out_features, in_features)*.01
        self.b = np.zeros((out_features, 1))

    def forward(self, x):
        return x @ self.W.T + self.b.T

In [2]:
class ReLU:

    def forward(self, x):
        return np.maximum(0, x)

In [3]:
class Sigmoid:

    def forward(self, x):
        return 1 / (1 + np.exp(-x))

In [4]:
class DeepNetwork:

    def __init__(self):
        self.layers = [
            Linear(2, 8),
            ReLU(),

            Linear(8, 8),
            ReLU(),

            Linear(8, 1),
            Sigmoid()
        ]

    def forward(self, x):
        out = x

        for layer in self.layers:
            out = layer.forward(out)

        return out

In [8]:
X = np.array([
    [0.2, 0.5],
    [1.0, 0.3],
    [0.7, 0.8]
])

model = DeepNetwork()

y = model.forward(X)

print(y.shape)

(3, 1)


In [9]:
# Generalise arbitrary depth

class Sequential:

    def __init__(self, *layers):
        self.layers = list(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)

        return x

In [11]:
# This is like torch.nn.Sequential

model = Sequential(
    Linear(2, 16),
    ReLU(),
    Linear(16, 32),
    ReLU(),
    Linear(32, 16),
    ReLU(),
    Linear(16, 1),
    Sigmoid()
)

## 5. Backpropagation Through Deep Networks

- For every layer l-th, we need:

```math
\frac{\partial L}{\partial W^{(l)}}, \frac{\partial L}{\partial b^{(l)}}
```

- Instead of differentiating everything independently, we have **error signal**.

```math
\delta^{(l)} = \frac{\partial L}{\partial z^{(l)}}
```

By chain rule, at the output layer:

```math
\delta^{(L)} = \frac{\partial L}{\partial a^{(L)}} \odot f'(z^{(L)})
```

### Weight gradient

```math
z^{(l)} = W^{(l)}a^{(l-1)} + b^{(l)}
```

For a single neuron:

```math
z_i = \sum_j W_{ij}a_j + b_j
```

Therefore,

```math
\frac{\partial z_i}{\partial W_{ij}} = a_j
```

Chain rule:

```math
\frac{\partial L}{\partial W^{(l)}} = \delta^{(l)} (a^{(l-1)})^T
```

### Bias gradient

```math
\frac{\partial z}{\partial b} = 1 \Rightarrow \frac{\partial L}{\partial b^{(l)}} = \delta^{(l)}
```

### Generalisation

Hidden layers

```math
\frac{\partial L}{\partial z^{(l)}} = \frac{\partial L}{\partial z^{(l+1)}} \frac{\partial z^{(l+1)}}{\partial a^{(l)}} \frac{\partial a^{(l)}}{\partial z^{(l)}}
```

But,

```math
z^{(l+1)} = W^{(l+1)}a^{(l)} + b^{(l+1)} \Rightarrow \frac{\partial z^{(l+1)}}{\partial a^{(l)}} = (W^{(l+1)})
```

Therefore:

```math
\delta^{(l)} = (W^{(l+1)})^T \delta^{(l+1)} \odot f'(z^{(l)})
```

**Proof**

For:

```math
W = \begin{bmatrix}
    w_{11} & w_{12} & w_{13} \\
    w_{21} & w_{22} & w_{23}
\end{bmatrix} \\

z = \begin{bmatrix} z_1 \\ z_2 \end{bmatrix} = \begin{bmatrix}
    w_{11}a_1 + w_{12}a_2 + w_{13}a_3 + b_1 \\
    w_{21}a_1 + w_{22}a_2 + 2_{23}a_3 + b_2
\end{bmatrix}
```

Then:

```math
\frac{\partial z_1}{\partial a} = \begin{bmatrix} w_{11} & w_{12} & w_{13} \end{bmatrix} \\
\frac{\partial z_2}{\partial a} = \begin{bmatrix} w_{21} & w_{22} & w_{23} \end{bmatrix}
```

Stack them -> Jacobian:

```math
J_{ij} = \frac{\partial z_i}{\partial a_j} \\
\Rightarrow J = \begin{bmatrix}
    w_{11} & w_{12} & w_{13} \\
    w_{21} & w_{22} & w_{23}
\end{bmatrix} \\
\Rightarrow \frac{\partial z}{\partial a} = W
```

Under Jacobian convention:
- Rows = outputs
- Cols = inputs

But in backprop, using chain rule:

```math
\frac{\partial L}{\partial a} = (\frac{\partial z}{\partial a})^T \frac{\partial L}{\partial z} = W^T \frac{\partial L}{\partial z}
```

**Check**

- $a^{(l)}$ has 3 elements
- $z^{(l+1)}$ has 2 elements
- $W \in \mathbb{R}^{2 \times 3}$
- $\frac{\partial L}{\partial z^{(l+1)}} \in \mathbb{R}^{2 \times 1}$
- Since we have $a_1, a_2, a_3$, then $\frac{\partial L}{\partial a} \in \mathbb{R}^{3 \times 1}$
- shape = (3 \times 2)(2 \times 1)

### One thing to remember

> Don't mix **Jacobian** with **vector gradients**.

### Complete algorithm

Forward pass:

```math
a^{(o)} \rightarrow a^{(1)} \rightarrow ... \rightarrow a^{(L)}
```

Backward pass:

```math
\delta^{(L)} \rightarrow \delta^{(L-1)} \rightarrow ... \rightarrow \delta^{(1)}
```

In [12]:
import numpy as np

class Linear:

    def __init__(self, in_features, out_features):
        self.W = np.random.randn(out_features, in_features) * 0.01
        self.b = np.zeros((1, out_features))

    def forward(self, x):
        self.x = x
        return x @ self.W.T + self.b

    def backward(self, grad_output):
        # Gradient w.r.t. weights
        self.grad_W = grad_output.T @ self.x

        # Gradient w.r.t. biases
        self.grad_b = grad_output.sum(axis=0, keepdims=True)

        # Gradient for previous layer
        grad_input = grad_output @ self.W

        return grad_input

In [13]:
class ReLU:

    def forward(self, x):
        self.mask = x > 0
        return np.maximum(0, x)

    def backward(self, grad_output):
        return grad_output * self.mask

In [14]:
class Sequential:

    def __init__(self, *layers):
        self.layers = list(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)

        return x

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

        return grad

In [15]:
model = Sequential(
    Linear(2, 8),
    ReLU(),
    Linear(8, 1)
)

X = np.random.randn(5, 2)

output = model.forward(X)

# Assume mean squared error:
# L = 1/2 * ||output - target||^2

target = np.random.randn(5, 1)

grad_output = output - target

model.backward(grad_output)

array([[-1.42283005e-05,  3.80798593e-05],
       [-6.63671283e-05,  1.77621418e-04],
       [-5.79974662e-06,  1.55221304e-05],
       [-2.29237329e-05,  6.13518477e-05],
       [-3.68499372e-05,  9.86231930e-05]])

## 6. Jacobians and the Chain Rule in Matrix Form

### What is Jacobian?

For:

```math
x, y \in \mathbb{R}, y = f(x) \Rightarrow \frac{dy}{dx} \in \mathbb{R}
```

But now if we have $f: \mathbb{R}^2 \rightarrow \mathbb{R}$, then there isn't one derivative, there are two:

```math
\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}
```

packed into the gradient:

```math
\nabla f = \begin{bmatrix}
    \frac{\partial f}{\partial x_1} \\
    \frac{\partial f}{\partial x_2}
\end{bmatrix}
```

Now suppose output is also a vector: $f: \mathbb{R}^2 \rightarrow \mathbb{R}^3$, with $y_1 = x_1 + x_2$, $y_2 = x_1x_2$, $y_3 = x_1^2$.
- Ouput dim = 3
- Input dim = 2
- Number of partial derivatives = $3 \times 2 = 6$

> **Jacobian matrix** is where to store the derivatives.

```math
f: \mathbb{R}^n \rightarrow \mathbb{R}^m, \\
J_f(x) = \frac{\partial y}{\partial x} = \begin{bmatrix}
    \frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_1}{\partial x_n} \\
    \vdots & & \vdots \\
    \frac{\partial y_m}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{bmatrix}
```

- Rows = # outputs
- Cols = # inputs

### Why NNs need Jacobians?

Suppose $x \rightarrow y \rightarrow z$ with $x \in \mathbb{R}^n, y \in \mathbb{R}^m, z \in \mathbb{R}^k$.

Where $y = f(x)$ and $z = g(y)$. We want $\frac{\partial z}{\partial x}$.

For one output $z_i$,

```math
z_i = g_i(y) = g_i(f(x))
```

Then,

```math
\frac{\partial z_i}{\partial x_j} = \sum_{r=1}^m \frac{\partial z_i}{\partial y_r} \frac{\partial y_r}{\partial x_j}
```

If we stack every $(i,j)$, the summation becomes matrix multiplication.

```math
J_{z, x} = J_{z, y}J_{y, x}
```

e.g.,

```math
A = \begin{bmatrix}
    1 & 2 \\
    3 & 4
\end{bmatrix}, y = Ax
```

then $J_{y, x} = A$.

Similarly, if $z = By$, then $J_{z, y} = B$. Therefore:

```math
J_{z, x} = BA
```

> Forward propagation: $z = B(Ax)$, then in backprop, Jacobians multiply in exactly the same order.

### Jacobian of a Linear Layer

Given $z = Wa + b$, what is $\frac{\partial z}{\partial a}$?

For one element:

```math
z_i = \sum_{j}W_{ij}a_j + b_i
```

then $\frac{\partial z_i}{\partial a_j} = W_{ij}$. Thus,

```math
J_{z, a} = W
```

In NN's math, $\frac{\partial L}{\partial z}$ is treated as a **column vector**, so applying the chain rule:

```math
\frac{\partial L}{\partial a} = J_{z, a}^T \frac{\partial L}{\partial z} = W^T \frac{\partial L}{\partial z}
```

### Jacobian of ReLU

$a_i = \max(0, z_i)$. So $\frac{\partial a_i}{\partial z_j} = 0$ for $i \ne j$.

- Each output depends only on its own input.

```math
J = \begin{bmatrix}
    g'(z_1) & 0 & 0 \\
    0 & g'(z_2) & 0 \\
    0 & 0 & g'(z_3)
\end{bmatrix}
```

### Jacobian of Softmax

```math
s_i = \frac{e^{z_i}}{\sum_j e^{x_j}} \Rightarrow \frac{\partial s_i}{\partial z_j} \ne 0 \\
\Rightarrow J = \text{diag}(s) - ss^T
```

In [16]:
import numpy as np

def numerical_jacobian(f, x, eps=1e-6):
    """
    Compute Jacobian of f: R^n -> R^m
    """

    x = np.asarray(x, dtype=float)
    y = f(x)

    m = len(y)
    n = len(x)

    J = np.zeros((m, n))

    for j in range(n):
        dx = np.zeros_like(x)
        dx[j] = eps

        y_plus = f(x + dx)
        y_minus = f(x - dx)

        J[:, j] = (y_plus - y_minus) / (2 * eps)
    
    return J

In [17]:
def f(x):
    return np.array([
        x[0] + x[1],
        x[0] * x[1],
        x[0] ** 2
    ])

x = np.array([2., 3.])

print(numerical_jacobian(f, x))

[[1. 1.]
 [3. 2.]
 [4. 0.]]


## 7. Information Flow in Deep Networks

- Hierarchical representation learning.
- Representation > Better representation.
- Forward = info about the input. Input > prediction.
- Backward = info about the error. Loss > correction.
- Info can be comressed or expanded.

### Info preservation

- If W is invertible, info is preserved $x = W^{-1}y$.
- But if $\text{rank}(W) < n$, info disappears forever. (e.g., 3 dim -> 1 dim, impossible to recover perfectly)

### Mutual info

```math
I(X;Y)
```

How much information about $X$ is contained in $Y$.

- Large $I$ = Hidden layer remembers the input.
- Small $I$ = Hidden layer discarded information.

### Gradient as information

```math
\delta^{(l)} = \frac{\partial L}{\partial z^{(l)}}
```

tells which direction improves the prediction.

In [18]:
import numpy as np

class Dense:

    def __init__(self, in_dim, out_dim):
        self.W = np.random.randn(out_dim, in_dim)*.1
        self.b = np.zeros((out_dim, 1))

    def forward(self, x):
        return self.W @ x + self.b
    
def relu(x):
    return np.maximum(0, x)

In [19]:
layers = [
    Dense(4, 8),
    Dense(8, 8),
    Dense(8, 4),
    Dense(4, 2)
]

In [20]:
x = np.random.randn(4,1)

representation = x

for i, layer in enumerate(layers):

    representation = layer.forward(representation)
    representation = relu(representation)

    print(f"Layer {i+1}")
    print(representation.T)

Layer 1
[[0.         0.         0.         0.         0.2279499  0.10867903
  0.1390879  0.36046077]]
Layer 2
[[0.05207612 0.         0.00016234 0.         0.         0.
  0.         0.        ]]
Layer 3
[[2.43233685e-03 5.46694917e-03 8.99467660e-05 1.74575379e-03]]
Layer 4
[[0.00080669 0.        ]]


Notice:
- The vector changes every layer.
- The network is rewriting the representation.

In [22]:
previous = x

for layer in layers:
    current = relu(layer.forward(previous))
    distance = np.linalg.norm(current - previous[:current.shape[0]].T)

    print(distance)
    previous = current

5.658611942115423
1.2801296542999823
0.09993073034694254
0.00775658015921363


- Large distance -> Layer changes representation significantly
- Small distance -> Representation barely changes.

## 8. Capacity, Width, and Depth

- **Capacity**: How many different functions can a network represent?
- **Width**: How many neurons are in each layer?
- **Depth**: How many layers does a network have?

### Capacity

A linear model $y = wx + b$ only produces a straight line, never curve. That's capacity.

Now increase capacity, the model can represent more functions.

```math
y = W_2g(W_1x + b_1) + b_2
```

- Two ways to increase capacity:
  - Wide network
  - Deep network
- Capacity = Richness of the family: $\mathcal{F} = {f_{\theta}: \theta \in \Theta}$ (search for $\theta$ such that $f_{\theta}(x) \approx f^*(x)$)

### Width

- Number of neurons in a layer.
- Layer $l$ has width $n_l$.

### Depth

- Number of transformations.

```math
x \rightarrow f_1 \rightarrow f_2 \rightarrow ... \rightarrow f_L
```

- Depth = $L$
- Allow networks to reuse intermediate computations insead of recalculating them repeatedly.

### Param Count

Input = 100, Hidden = 200, Output = 50.

```math
\text{Parameters } = 200 \times 100 + 200 + 50 \times 200 + 50
```

Formula:

```math
\sum_{l=1}^L(n_ln_{l-1} + n_l)
```

### Universal Approximation Theorem

> A feedforward neural network with one sufficiently wide hidden layer and a suitable activation function can approximate any continuous function on a compact domain to arbitrary accuracy.

### Conclusion

- **Width** increases the number of features a layer can compute in parallel.
- **Depth** increases the number of sequential transformation applied to those features.

In [23]:
import numpy as np

class Dense:

    def __init__(self, in_dim, out_dim):
        self.W = np.random.randn(out_dim, in_dim)*.1
        self.b = np.zeros((out_dim, 1))

    @property
    def parameters(self):
        return self.W.size + self.b.size

In [24]:
layers = [
    Dense(2, 16),
    Dense(16, 16),
    Dense(16, 1)
]

total = sum(layer.parameters for layer in layers)
print(f"Total # params: {total}")

Total # params: 337


In [28]:
# Generalise

def param_count(layer_sizes):
    total = 0

    for i in range(len(layer_sizes) - 1):
        total += layer_sizes[i] * layer_sizes[i+1]
        total += layer_sizes[i+1]
    return total

In [29]:
param_count([784,128,64,10])

109386

In [ ]:
import torch

model = torch.nn.Sequential()

# param count
sum(p.numel() for p in model.parameters())

## 9. Deep Network Architectures from Scratch

Composition:

```math
a^{(L)} = f^{(L)} \circ f^{(L - 1)} \circ ... \circ f^{(1)}(x)
```

where each

```math
f^{(l)}(a) = \sigma(W^{(l)}a + b^{(l)})
```

### Matrix dimensions

e.g.,
- Input: 5 features
- Hidden1: 8 neurons
- Hidden2: 6 neurons
- Output: 2 neurons

then $W_1 \in \mathbb{R}^{8 \times 5}, W_2 \in \mathbb{R}^{6 \times 8}, W_3 \in \mathbb{R}^{2 \times 6}$.

Architecture notation

```math
5 \rightarrow 8 \rightarrow 6 \rightarrow 2
```

### Universal forward equation

For $l = 1,..., L$

```math
z^{(l)} = W^{(l)}a^{(l-1)} + b^{(l)} \\
a^{(l)} = f^{(l)}(z^{(l)})
```

In [41]:
import numpy as np

class Dense:

    def __init__(self, in_features, out_features):
        self.W = np.random.randn(out_features, in_features)*.01
        self.b = np.zeros((out_features, 1))

    def forward(self, x):
        return self.W @ x + self.b
    
class ReLU:

    def forward(self, x):
        return np.maximum(0, x)
    
class Sigmoid:

    def forward(self, x):
        return 1 / (1 + np.exp(-x))
    

class Sequential:

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

In [42]:
model = Sequential([

    Dense(4,16),
    ReLU(),

    Dense(16,8),
    ReLU(),

    Dense(8,1),
    Sigmoid()

])

In [43]:
x = np.random.randn(4,1)
y = model.forward(x)

print(y)

[[0.49999819]]


## 10. Building an L-Layer Neural Network in NumPy

We want:

```python
model = NeuralNetwork(...)
y_hat = model.forward(X)
loss = model.loss(y_hat, Y)
model.backward(Y)
model.step(lr)
```

- Every layer = 1 object
- Each layer stores exacly the same information:
  - Weights
  - Bias
  - Acitivation
  - Gradient
- A NN = a collection of layers.
- Each layer should know how to:
  - forward
  - backward
  - store gradients
- Each NN should know how to:
  - run forward
  - run backward
  - update parameters

### Forward propagation

```math
a^{(0)} \rightarrow a^{(1)} \rightarrow ... \rightarrow a^{(L)}
```

- Each layer stores:
  - Input activation
  - Pre-activation
  - Output activation

### Backward propagation

Final gradient:

```math
\delta^{(L)} = \frac{\partial L}{\partial z^{(L)}}
```

Layer $L$ computes:

```math
dW^{(L)} = \delta^{(L)}(a^{(L-1)})^T
db^{(L)} = \sum \delta^{(L)}
```

then

```math
\delta^{(l - 1)} = (W^{(l)})^T\delta^{(l)} \odot f'(z^{(l-1)})
```

### Param update

```math
W \leftarrow W - \eta dW \\
b \leftarrow b - \eta db
```

In [44]:
import numpy as np

class Dense:

    def __init__(self, in_features, out_features):
        self.W = np.random.randn(out_features, in_features) * 0.01
        self.b = np.zeros((out_features, 1))

        self.dW = None
        self.db = None

        self.input = None
        self.output = None

    def forward(self, x):
        self.input = x
        self.output = self.W @ x + self.b
        return self.output

In [45]:
class ReLU:

    def __init__(self):
        self.input = None

    def forward(self, x):
        self.input = x
        return np.maximum(0, x)

In [46]:
class Sigmoid:

    def __init__(self):
        self.output = None

    def forward(self, x):
        self.output = 1/(1+np.exp(-x))
        return self.output

In [47]:
class NeuralNetwork:

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)

        return x

In [50]:
class NeuralNetwork:
    ...

    def parameters(self):
        params = []

        for layer in self.layers:
            if hasattr(layer, "W"):
                params.append(layer)

        return params
    
    def step(self, lr):
        for layer in self.parameters():
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db

In [ ]:
# Test
# 4 -> 32 -> 16 -> 1
# Column vector
# Input: X (4, m)
# Layer 1 (32, 4)

model = NeuralNetwork([

    Dense(4,32),
    ReLU(),

    Dense(32,16),
    ReLU(),

    Dense(16,1),
    Sigmoid()

])

In [52]:
# Torch comparison
import torch.nn as nn

torch_model = nn.Sequential(
    nn.Linear(4, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
    nn.Sigmoid()
)

## 11. Computational Complexity of Deep Networks

- One neuron, $x, w \in \mathbb{R}^d$, then $w^Tx$ requires:
  - d multiplication
  - d - 1 addition
  - total O(d)
- Suppose this layer has 1000 neurons -> O(1000d) operations
- Next layer has 10000 neurons -> O(1000 * 10000 * d) operations.
- Deep learning complexity = Edge complexity.
- 3 types of complexity:
  - **Time**: FLOPs, multiply-adds
  - **Memory**: RAM, VRAM (for params, activations, gradients, optimiser states)
  - **Communication**: For distributed training between GPUs (especially modern models)

### Complexity math

One layer:

```math
z^{(l)} = W^{(l)}a^{(l-1)} + b^{(l)}
```

Then complexity:
- One sample: $O(n_ln_{l-1})$ for multiplication

Suppose activation function = ReLU

```math
a_i = \max(0, z_i)
```

- activation complexity = $O(n_l)$
- Total: $O(n_ln_{l-1})$

Complexity of a whole network of $L$ layers:

```math
O(\sum_{l=1}{L}n_ln_{l-1})
```

### Mini-batch

Suppose Batch size = $B, A \in \mathbb{R}^{n_{l-1} \times B}$

```math
Z = WA + b
```

then:
- forward cost = $O(Bn_ln_{l-1})$

Backprop:

```math
\delta^{(l)} = (W^{(l+1)})^T\delta^{(l + 1)} \circ \sigma'(z^{(l)})
```

then:
- backward cost = $O(n_ln_{l+1})$

Gradient: $\frac{\partial L}{\partial W} = \delta a^T$ is also multiplication.
- gradient cost = $O(n_ln_{l-1})$

### Parammeter complexity

$W^{(l)}$ contains $n_ln_{l-1}$ numbers. Biases $b^{(l)}$ contains $n_l$ numbers, then number of params is:

```math
P = \sum_l(n_ln_{l-1} + n_l)
```

Because $n_ln_{l-1} \gg n_l$, so:

```math
P \approx \sum n_ln_{l-1}
```

### Acitivation complexity

During backprop, store every activation. So memory: $O(B\sum n_l)$

### Total memory

Memory consists of:
- params
- activations
- gradients
- optimisation state

for Adam, optimiser state stores:
- momentum
- variance

> Total roughly 3x of parameters

In [ ]:
import numpy as np

class ComplexityAnalyser:

    def __init__(self, layer_sizes):
        self.layers = layer_sizes

    def param_count(self):
        total = 0

        for i in range(len(self.layers) - 1):
            total += self.layers[i] * self.layers[i+1]
            total += self.layers[i+1]

        return total
    
    def forward_flops(self, batch_size=1):
        """
        Floating point operations
            - Multiplication
            - Addition
        """
        flops = 0

        for i in range(len(self.layers) - 1):
            flops += batch_size * self.layers[i] * self.layers[i+1]
        
        return flops
    
    def backward_flops(self, batch_size=1):
        """
        O(n_l*n_l-1) for backprop
        O(n_l*n_l-1) for gradient
        => TOTAL = 2*n_l*n_l-1 flops.
        """
        return 2 * self.forward_flops(batch_size)
    
    def training_flops(self, batch_size=1):
        return (
            self.forward_flops(batch_size)
            + self.backward_flops(batch_size)
        )
    
    def summary(self, batch_size=32):
        print("Architecture:", self.layers)
        print("Parameters:", self.param_count())
        print("Forward FLOPs:", self.forward_flops(batch_size))
        print("Backward FLOPs:", self.backward_flops(batch_size))
        print("Training FLOPs:", self.training_flops(batch_size))

In [62]:
net = ComplexityAnalyser([784,512,256,128,10])

net.summary(batch_size=64)

Architecture: [784, 512, 256, 128, 10]
Parameters: 567434
Forward FLOPs: 36257792
Backward FLOPs: 72515584
Training FLOPs: 108773376


## 12. Why Training Deep Networks Is Difficult

Why not just adding more layers?

Five fundamental difficulties:
1. Vanishing gradients
2. Exploding gradients
3. Poor conditioning of optimisation
4. Internal covariate shift (historical motivation)
5. Signal propagation through deep networks

A few math:

```math
\delta^{(1)} = (W^{(2)})^T\delta^{(2)} \circ \sigma'(z^{(2)}) \\
\delta^{(1)} = (W^{(2)})^T((W^{(3)})^T \delta^{(3)} \circ \sigma'(z^{(3)})) \circ \sigma'(z^{(2)})
```

General case:

```math
\delta^{(1)} = (\prod_{l=2}^L(W^{(l)})^TD^{(l-1)})\delta^{(L)}
```

where $D^{(l)} = \text{diag}(\sigma'(z^{(l)}))$

### Vanishing gradients

- Gradient scaled down to 0 (e.g., every layer scales vectors by 0.5)
- Sigmoid, $\sigma'(x) = \sigma(x)(1 - \sigma(x)) \le .25$
- ReLU preserves gradient magnitude.

### Exploding gradients

- Gradient get too big (e.g., every layer scales vectors by 2)

### Spectral viewpoint

Given:

```math
\delta_{l-1} = W^T\delta_l
```

Cauchy-Schwarz:

```math
||W^T\delta|| \le ||W||_2||\delta||
```

- $|W||_2 < 1$: gradients shrink
- $|W||_2 > 1$: gradients grow

## 13. Case Study: Learning XOR with Depth

For $x_1, x_2 \in \{0, 1\}$, we have to predict:

| $x_1$ | $x_2$ | XOR |
|-------|-------|-----|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

with 

```math
\^{y} = \sigma (w_1x_1 + w_2x_2 + b)
```

the neuron draws a straight line: $w_1x_1 + w_2x_2 + b = 0$.

- Both positives and negatives are on opposite corners.

> XOR is not linearly separable.

### Why one neuron cannot solve XOR?

```math
f(x) = w^Tx + b
```

- Positives: $(1, 0), (0, 1)$
- Negatives: $(0, 0), (1, 1)$

Positives, let:

```math
f(1, 0) = w_1 + b > 0 \\
f(0, 1) = w_2 + b > 0 \\
\Rightarrow w_1 + w_2 + 2b > 0 \Leftrightarrow w_1 + w_2 + b > -b > 0
```

Negatives:

```math
f(0, 0) = b < 0 \\
f(1, 1) = w_1 + w_2 + b < 0
```

Then we have $w_1 + w_2 + b$ both > 0 and < 0 (Contradiction).

### Why hidden neurons solve XOR?

- XOR: (A or B) but not (A and B )

```math
(A \vee B) \wedge \neg(A \wedge B)
```

Logically, if $h_1 = A \vee B, h_2 = A \wedge B$, then the output has form: $h_1 - h_2$.

Mathematically, choose $w = [1, 1]$.

OR:
- (0, 0) -> 0
- (1, 0) -> 1
- (0, 1) -> 1
- (1, 1) -> 1

then:

```math
h1 = x_1 + x_2 - 0.5
```

AND:
- (0, 0) -> 0
- (1, 0) -> 0
- (0, 1) -> 0
- (1, 1) -> 1

then:

```math
h_2 = x_1 + x_2 - 1.5
```

Output neuron:
- OR = 1, AND = 0 -> 1
- OR = 1, AND = 1 -> 0

choose:

```math
z = h_1 - 2h_2 - 0.5
```

**Mat Mul form**

```math
W_1 = \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}, 
x = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}, b_1 = \begin{bmatrix} -0.5 \\ -1.5 \end{bmatrix}
```

Activation:

```math
h = \sigma(W_1x + b_1)
```

Output:

```math
W_2 = \begin{bmatrix} 1 \\ -2 \end{bmatrix},
b_2 = -0.5 \\
\^{y} = \sigma(W_2h +b_2)
```

In [63]:
import numpy as np

x = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float16)

y = np.array([[0], [1], [1], [0]], dtype=np.float16)

In [64]:
# Hard-threshold netwok

def step(x):
    return (x > 0).astype(np.float16)

In [65]:
W_1 = np.array([
    [1, 1],
    [1, 1]
], dtype=np.float16)
b_1 = np.array([-.5, -1.5], dtype=np.float16)

W_2 = np.array([[1, -2]], dtype=np.float16)
b_2 = np.array([-.5], dtype=np.float16)

In [67]:
H = step(x @ W_1.T + b_1)
y = step(H @ W_2.T + b_2)
print(y)

[[0.]
 [1.]
 [1.]
 [0.]]


In [68]:
# Sigmoid

def sigmoid(x):
    return 1/(1+np.exp(-x))


# Increase weight magnitude

scale = 20

W1 = scale*np.array([
    [1,1],
    [1,1]
])

b1 = scale*np.array([-0.5,-1.5])

W2 = scale*np.array([[1,-2]])
b2 = scale*np.array([-0.5])


# Forward

H = sigmoid(X @ W1.T + b1)
Y = sigmoid(H @ W2.T + b2)
print(np.round(Y,4))

[[3.000e-04]
 [0.000e+00]
 [9.991e-01]
 [9.999e-01]
 [0.000e+00]]


## 14. Deep Learning Framework Design

- Repeated abstraction
- Interface principle: input -> output

In [ ]:
class Layer:

    def forward(self, x):
        return NotImplementedError
    
    def backward(self, grad):
        return NotImplementedError

In [ ]:
class Sequential:

    def __init__(self, *layers):
        ...
    
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)

        return x

In [ ]:
# torch version of Sequential

from torch.nn import Sequential

model = Sequential(
    Linear(5, 10),
    Sigmoid(),
    Linear(10, 2)
)

Y = model.forward(np.random.randn(16, 5))

assert Y.shape == (16, 2)

## 15. Research Perspective: Depth as Hierarchical Computation

- **Universal Approximation Theorem**
- Pixels -> How to know it's a cat.
- Not checking 1,000,000 pixels. We detect edges -> shapes -> eyes -> ears -> cat. Each layer builds more abstract information.
- Analogy: `renderGame()` is a composition of:
  - `drawPlayer()`
  - `drawEnemy()`
  - `drawMap()`
  - `drawWeapon()`
- No relearn, reuse.
- Computational graphs.